# Reconstruct a Z candidate from two muons

In 45 minutes, select opposite-sign muons and plot their invariant mass. Generate $\gamma^*/Z$ Drell–Yan events at 13 TeV with a 60 GeV lower cut on the hard-process mass and the decay channel restricted to muons. The plot uses generator-level final muons after any radiation.

Require $p_T>20$ GeV and $|\eta|<2.4$ for both muons. PDG ID 13 is $\mu^-$ and −13 is $\mu^+$. Keep the highest-$p_T$ muon of each sign and form at most one candidate per event. In events with extra muons this rule can pair a muon from another source.

Sum the four-vectors in units with $c=1$:

$$
m_{\mu\mu}=\sqrt{(E_1+E_2)^2-(p_{x1}+p_{x2})^2-(p_{y1}+p_{y2})^2-(p_{z1}+p_{z2})^2}.
$$

In [ ]:
import matplotlib.pyplot as plt
from hep_lab.kinematics import invariant_mass
from hep_lab.simulation import generate

def four_vector(particle):
    return particle.energy, particle.px, particle.py, particle.pz

def candidate_mass(event, min_pt=20):
    muons = [p for p in event if abs(p.pdg_id) == 13 and p.pt > min_pt and abs(p.eta) < 2.4]
    negative = max((p for p in muons if p.pdg_id == 13), key=lambda p: p.pt, default=None)
    positive = max((p for p in muons if p.pdg_id == -13), key=lambda p: p.pt, default=None)
    if negative is None or positive is None:
        return None
    return invariant_mass(four_vector(negative), four_vector(positive))

events = generate("dimuon", n_events=1200, seed=42)
masses = [mass for event in events if (mass := candidate_mass(event)) is not None]
inside = sum(80 <= mass <= 100 for mass in masses)
print(f"{len(masses)}/{len(events)} events selected; {inside} candidates in 80–100 GeV")

The resonance is near 91 GeV. The shape includes the virtual-photon contribution, interference, radiation and the influence of the selection. The dashed line marks a reference value. The notebook performs no mass or width fit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.hist(masses, bins=40, range=(50, 130), histtype="stepfilled", color="#007580", alpha=0.8)
ax.axvline(91.2, color="#a43d27", linestyle="--", label="Z mass reference")
ax.set(xlabel=r"$m_{\mu\mu}$ [GeV]", ylabel="Candidates / 2 GeV", title="Simulated dimuon mass")
ax.legend()
plt.show()
print(f"Outside plotted range: {sum(not 50 <= mass <= 130 for mass in masses)}")

## Challenges

### Physics
Explain why a longitudinal boost changes individual muon energies while preserving the pair mass. Give one mechanism that can put the reconstructed pair below the Z pole. **Hint** Consider the Minkowski norm of the sum and a photon outside the two selected vectors.

### Coding
Repeat with a 30 GeV threshold on the same events. Report the selected fraction for each threshold and overlay the mass distributions with identical bin edges. **Hint** Pass the threshold into `candidate_mass`.

### Codex
Ask Codex to draft analytic tests for mass, vector swapping and same-sign rejection. Compute the expected values by hand and run the tests. **Hint** For the mass alone, try two back-to-back 10 GeV massless vectors.